# Final Test Evaluation — Social Science Concept Integration

**Purpose:** This notebook performs the definitive held-out test evaluation of ML models for concept harmonisation in social science.  
Each model is evaluated under **three techniques** (Clustering, Pairwise, Seeded Clustering) and subjected to **five psychometric audits**:

| # | Audit | What it measures |
|---|-------|-----------------|
| 1 | **Reliability** | Standard Error of Model (SEM) under stochastic embedding noise |
| 2 | **Discriminant Validity** | False-positive rate on lexically similar negative pairs |
| 3 | **DIF (Rare-Word Bias)** | Recall gap between common and rare terminology |
| 4 | **Semantic Decay** | Recall degradation as keyword overlap vanishes |
| 5 | **Structural Validity** | Correlation with expert ELSST/APA graph distances (Spearman, Pearson, Biserial) |

> **Reproducibility:** All random operations use `SEED = 42`. Models are loaded from local HuggingFace cache. Best hyperparameters are fixed from prior validation.

## 0. Environment Setup & Imports

In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*n_jobs value 1 overridden.*")
import sys
import os
from pathlib import Path
import importlib

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from contextlib import redirect_stdout
from io import StringIO
from IPython.display import display, Markdown

# Local project modules
from scripts import model_utils_shared as shared
from scripts import model_utils_clustering as muc
from scripts import model_utils_pairwise as mup
from scripts import model_utils_seed as mus
from scripts import config

# Reload config to get latest changes
importlib.reload(config)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Project root    : {config.PROJECT_ROOT}")
print(f"Random seed     : {config.SEED}")

PyTorch version : 2.11.0+cpu
CUDA available  : False
Project root    : C:\Users\norouzin\OneDrive - Tilburg University\Desktop\Third_paper\Social_Science_Construct_Harmonization
Random seed     : 42


In [ ]:
# ── Export paths ──────────────────────────────────────────────────────────────
import os
RESULTS_DIR = 'results/elsst/'
os.makedirs(RESULTS_DIR, exist_ok=True)


## 1. Configuration — Best Hyperparameters (from Cross-Validation)

These parameters were determined during the training phase via grid search / cross-validation.  
They are **fixed** for this final held-out evaluation and must **not** be modified.

In [ ]:
# ── Models to evaluate ────────────────────────────────────────────────────────
MODELS_TO_EVAL = [
    "all-mpnet-base-v2",
    "dwulff/mpnet-personality",
    "allenai/scibert_scivocab_uncased",
    "bert-base-uncased",
]

# ── Best HDBSCAN clustering parameters (from callibration grid search) ───────────
BEST_CLUSTERING_PARAMS = config.BEST_CLUSTERING_PARAMS
# ── Best pairwise cosine similarity thresholds ───────────────────────────────
BEST_PAIRWISE_THRESHOLDS = config.BEST_PAIRWISE_THRESHOLDS

# ── Best seeded clustering parameters ────────────────────────────────────────
BEST_SEEDED_PARAMS = config.BEST_SEEDED_PARAMS

# ── ELSST Data paths (held-out test set) ───────────────────────────────────────────
TEST_POS_PATH = "datasets/processed_datasets/elsst/test_positive_pairs.csv"
TEST_NEG_PATH = "datasets/processed_datasets/elsst/test_negative_pairs.csv"

# ── Audit parameters ────────────────────────────────────────────────────────
NOISE_LEVELS = config.NOISE_LEVELS
N_RELIABILITY_RUNS = config.N_RELIABILITY_RUNS

print("✓ Configuration loaded.")

✓ Configuration loaded.


## 2. Load Test Data & Prepare Indices

We load the held-out test set (positive + negative pairs) and build the index structures needed for all downstream techniques.

In [ ]:
shared.setup_reproducibility(config.SEED)

# Load test data
pos_df, neg_df, full_df = shared.load_test_data(TEST_POS_PATH, TEST_NEG_PATH)

display(Markdown(f"""
| Split | Count |
|-------|------:|
| Positive pairs | {len(pos_df):,} |
| Negative pairs | {len(neg_df):,} |
| **Total**      | **{len(full_df):,}** |
"""))

# Build index arrays
terms1 = full_df["term1"].astype(str).tolist()
terms2 = full_df["term2"].astype(str).tolist()
labels = full_df["label"].astype(int).values
shortest_path = full_df.get("shortest_path", pd.Series([-1] * len(full_df))).values

unique_terms = pd.unique(full_df[["term1", "term2"]].values.ravel("K")).tolist()
term_to_idx = {t: i for i, t in enumerate(unique_terms)}

idx1 = np.array([term_to_idx[t] for t in terms1], dtype=np.int32)
idx2 = np.array([term_to_idx[t] for t in terms2], dtype=np.int32)
idx1_t = torch.tensor(idx1, dtype=torch.long)
idx2_t = torch.tensor(idx2, dtype=torch.long)

pos_len = len(pos_df)

print(f"Unique terms: {len(unique_terms):,}")
print(f"Pairs with expert shortest_path ≥ 0: {(shortest_path >= 0).sum():,}")

Loading test datasets...



| Split | Count |
|-------|------:|
| Positive pairs | 1,557 |
| Negative pairs | 530,219 |
| **Total**      | **531,776** |


Unique terms: 1,876
Pairs with expert shortest_path ≥ 0: 241,280


In [ ]:
# ── Optional debug subset (remove this cell for full analysis) ───────────────
# Set FAST_DEBUG_MODE = True to run quickly on a very small sample.
# Remove this entire cell to restore full-dataset behavior with no other edits.
FAST_DEBUG_MODE = True
FAST_DEBUG_POS_N = 150
FAST_DEBUG_NEG_N = 150

if FAST_DEBUG_MODE:
    _p = min(FAST_DEBUG_POS_N, len(pos_df))
    _n = min(FAST_DEBUG_NEG_N, len(neg_df))
    pos_df = pos_df.sample(_p, random_state=config.SEED).reset_index(drop=True)
    neg_df = neg_df.sample(_n, random_state=config.SEED).reset_index(drop=True)
    full_df = pd.concat([pos_df, neg_df], ignore_index=True)

    terms1 = full_df["term1"].astype(str).tolist()
    terms2 = full_df["term2"].astype(str).tolist()
    labels = full_df["label"].astype(int).values
    shortest_path = full_df.get("shortest_path", pd.Series([-1] * len(full_df))).values

    unique_terms = pd.unique(full_df[["term1", "term2"]].values.ravel("K")).tolist()
    term_to_idx = {t: i for i, t in enumerate(unique_terms)}

    idx1 = np.array([term_to_idx[t] for t in terms1], dtype=np.int32)
    idx2 = np.array([term_to_idx[t] for t in terms2], dtype=np.int32)
    idx1_t = torch.tensor(idx1, dtype=torch.long)
    idx2_t = torch.tensor(idx2, dtype=torch.long)

    pos_len = len(pos_df)

    print(f"FAST_DEBUG_MODE=True -> sampled {len(pos_df):,} positives + {len(neg_df):,} negatives")
    print("Remove this cell for full analysis.")
else:
    print("FAST_DEBUG_MODE=False -> using full dataset")


## 3. Precompute Audit Masks

Build boolean masks for the lexical, frequency, and difficulty subsets used by Audits 2–5.  
These are computed once and reused across all models and techniques.

In [4]:
# Letter n-gram cache (3-grams) for lexical similarity
token_cache = {t: shared.letter_ngrams(t, n=3) for t in unique_terms}

# ── Audit 2: Lexical trap masks (hard & easy negatives) ──────────────────────
# Compute Jaccard similarity for all negative pairs
jacc_neg = np.array([
    shared.jaccard_sim(token_cache.get(t1, set()), token_cache.get(t2, set()))
    for t1, t2 in zip(neg_df["term1"], neg_df["term2"])
])

# Hard negatives: lexical traps (high surface similarity, Jaccard > 0.5)
hard_neg_mask_local = jacc_neg > 0.5
hard_neg_mask = np.zeros(len(full_df), dtype=bool)
hard_neg_mask[pos_len:] = hard_neg_mask_local

# Easy negatives: clearly distinct pairs (zero lexical overlap, Jaccard = 0.0)
easy_neg_mask_local = jacc_neg == 0.0
easy_neg_mask = np.zeros(len(full_df), dtype=bool)
easy_neg_mask[pos_len:] = easy_neg_mask_local

# Backward compatibility: lex_mask = hard_neg_mask
lex_mask = hard_neg_mask

# ── Audit 5: Easy / Hard positive masks ──────────────────────────────────────
jacc_pos = np.array([
    shared.jaccard_sim(token_cache.get(t1, set()), token_cache.get(t2, set()))
    for t1, t2 in zip(pos_df["term1"], pos_df["term2"])
])
easy_pos_mask = np.zeros(len(full_df), dtype=bool)
hard_pos_mask = np.zeros(len(full_df), dtype=bool)
easy_pos_mask[:pos_len] = jacc_pos > 0.5
hard_pos_mask[:pos_len] = jacc_pos == 0.0

# Backward compatibility
easy_mask = easy_pos_mask
hard_mask = hard_pos_mask

# ── Audit 4: Rare / Common word frequency masks ─────────────────────────────
try:
    from wordfreq import zipf_frequency

    term_freq = {}
    for t in unique_terms:
        toks = shared.simple_tokens(t)
        term_freq[t] = float(np.mean([zipf_frequency(tok, "en") for tok in toks])) if toks else 0.0

    pos_pair_freq = np.array([
        (term_freq.get(t1, 0.0) + term_freq.get(t2, 0.0)) / 2.0
        for t1, t2 in zip(pos_df["term1"], pos_df["term2"])
    ])
    low_thr = np.quantile(pos_pair_freq, 0.10)
    high_thr = np.quantile(pos_pair_freq, 0.90)

    rare_mask = np.zeros(len(full_df), dtype=bool)
    common_mask = np.zeros(len(full_df), dtype=bool)
    rare_mask[:pos_len] = pos_pair_freq <= low_thr
    common_mask[:pos_len] = pos_pair_freq >= high_thr
    has_wordfreq = True
except ImportError:
    rare_mask = common_mask = np.zeros(len(full_df), dtype=bool)
    has_wordfreq = False
    print("⚠ wordfreq not installed — Audit 4 will be skipped.")

# ── Summary ──────────────────────────────────────────────────────────────────
display(Markdown(f"""
| Audit Mask | N |
|-----------|--:|
| **Negatives** | |
| → Hard negatives (lexical traps, Jaccard > 0.5) | {hard_neg_mask.sum():,} |
| → Easy negatives (zero overlap, Jaccard = 0.0) | {easy_neg_mask.sum():,} |
| **Positives** | |
| → Easy positives (Jaccard > 0.5) | {easy_pos_mask.sum():,} |
| → Hard positives (Jaccard = 0.0) | {hard_pos_mask.sum():,} |
| **Frequency** | |
| → Common pairs (top 10 % freq) | {common_mask.sum():,} |
| → Rare pairs (bottom 10 % freq) | {rare_mask.sum():,} |
"""))



| Audit Mask | N |
|-----------|--:|
| **Negatives** | |
| → Hard negatives (lexical traps, Jaccard > 0.5) | 65 |
| → Easy negatives (zero overlap, Jaccard = 0.0) | 434,190 |
| **Positives** | |
| → Easy positives (Jaccard > 0.5) | 152 |
| → Hard positives (Jaccard = 0.0) | 517 |
| **Frequency** | |
| → Common pairs (top 10 % freq) | 156 |
| → Rare pairs (bottom 10 % freq) | 156 |


## 4. Load Models & Build Embeddings

Each model is loaded once and its embeddings are cached in memory.  
All subsequent technique evaluations and audits reuse these cached embeddings.

In [5]:
embedding_cache = {}

for model_name in MODELS_TO_EVAL:
    model_cfg = config.MODELS[model_name]
    model_type = model_cfg.get("type", config.TYPE_SENTENCE)
    display_name = model_cfg.get("display_name", model_name)

    print(f"\n{'─' * 60}")
    print(f"  {display_name}  ({model_name})")
    print(f"{'─' * 60}")

    model = shared.load_model(model_name=model_name, model_type=model_type)
    if model is None:
        print(f"  ✗ FAILED — skipping")
        continue

    embedding_cache[model_name] = shared.build_embeddings(model, unique_terms, batch_size=config.BATCH_SIZE)
    print(f"  ✓ Embeddings cached — shape {embedding_cache[model_name]['norm_np'].shape}")

    # Free model from GPU memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'═' * 60}")
print(f"  Models loaded: {len(embedding_cache)} / {len(MODELS_TO_EVAL)}")
print(f"{'═' * 60}")


────────────────────────────────────────────────────────────
  All-MPNet-Base-v2  (all-mpnet-base-v2)
────────────────────────────────────────────────────────────
Loading Model (all-mpnet-base-v2)...
  Encoding 1876 unique terms...


Batches: 100%|██████████| 118/118 [00:09<00:00, 12.32it/s]


  ✓ Embeddings cached — shape (1876, 768)

────────────────────────────────────────────────────────────
  MPNet-Personality  (dwulff/mpnet-personality)
────────────────────────────────────────────────────────────
Loading Model (dwulff/mpnet-personality)...
  Encoding 1876 unique terms...


Batches: 100%|██████████| 118/118 [00:22<00:00,  5.13it/s]


  ✓ Embeddings cached — shape (1876, 768)

────────────────────────────────────────────────────────────
  SciBERT (SciVocab)  (allenai/scibert_scivocab_uncased)
────────────────────────────────────────────────────────────
Loading Model (allenai/scibert_scivocab_uncased)...
  Encoding 1876 unique terms...


Batches: 100%|██████████| 118/118 [00:09<00:00, 11.95it/s]


  ✓ Embeddings cached — shape (1876, 768)

────────────────────────────────────────────────────────────
  BERT Base  (bert-base-uncased)
────────────────────────────────────────────────────────────
Loading Model (bert-base-uncased)...
  Encoding 1876 unique terms...


Batches: 100%|██████████| 118/118 [00:08<00:00, 13.98it/s]

  ✓ Embeddings cached — shape (1876, 768)

════════════════════════════════════════════════════════════
  Models loaded: 4 / 4
════════════════════════════════════════════════════════════


---

## 5. Final Evaluation — Three Techniques

We evaluate every model under three harmonisation techniques using their best hyperparameters from cross-validation.  
All predictions are stored in `preds_store` for reuse in the audit sections.

In [6]:
# Storage for all predictions (used by audits later)
preds_store = {"clustering": {}, "pairwise": {}, "seeded": {}}

### 5.1 Clustering (UMAP + HDBSCAN)

Uses `model_utils_clustering.run_hdbscan_clustering()` and `model_utils_clustering.evaluate_clustering()` with the best parameters.

In [7]:

# ── 5.1  CLUSTERING ──────────────────────────────────────────────────────────
clustering_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    emb = embedding_cache[model_name]
    params = BEST_CLUSTERING_PARAMS[model_name]

    # Dimensionality reduction via muc.reduce_embeddings_with_umap()
    n_comp = params.get("n_components")
    if n_comp is None or n_comp >= emb["norm_np"].shape[1]:
        reduced = emb["norm_np"]
    else:
        reduced, _ = muc.reduce_embeddings_with_umap(
            emb["norm_np"], n_components=n_comp, random_state=config.SEED
        )

    # Cluster via muc.run_hdbscan_clustering()
    cluster_labels = muc.run_hdbscan_clustering(
        reduced,
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
    )

    # Evaluate via muc.evaluate_clustering() — returns sklearn metrics + cluster stats
    p, r, f1, pos_acc, neg_acc, n_clusters, n_noise = muc.evaluate_clustering(
        cluster_labels, full_df, np.array(unique_terms)
    )

    # Also extract raw predictions for audit reuse
    l1 = cluster_labels[idx1]
    l2 = cluster_labels[idx2]
    preds = ((l1 != -1) & (l1 == l2)).astype(int)

    clustering_rows.append({
        "Model": config.MODELS[model_name]["display_name"],
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
        "Clusters": n_clusters, "Noise": n_noise,
    })

    preds_store["clustering"][model_name] = {
        "preds": preds, "cluster_labels": cluster_labels,
        "idx1": idx1, "idx2": idx2,
    }

display(Markdown("### Results — Clustering"))
display(pd.DataFrame(clustering_rows).set_index("Model"))


### Results — Clustering

,Precision,Recall,F1,Pos Acc,Neg Acc,Clusters,Noise
Model,,,,,,,
All-MPNet-Base-v2,0.7016,0.5029,0.5859,0.5029,0.9994,492,472
MPNet-Personality,0.6845,0.4920,0.5725,0.4920,0.9993,476,487
SciBERT (SciVocab),0.6099,0.2922,0.3951,0.2922,0.9995,421,724
BERT Base,0.4820,0.2832,0.3568,0.2832,0.9991,399,755


In [ ]:
# ── Export C: cluster size distribution ───────────────────────────────────────
_sz_rows = []
for model_name in MODELS_TO_EVAL:
    info = preds_store["clustering"].get(model_name)
    if info is None:
        continue
    cl = info["cluster_labels"]
    n_noise = int((cl == -1).sum())
    n_total = int(len(cl))
    unique_cl, counts = np.unique(cl[cl != -1], return_counts=True)
    for cid, sz in zip(unique_cl, counts):
        _sz_rows.append({
            "model": config.MODELS[model_name]["display_name"],
            "cluster_id": int(cid),
            "cluster_size": int(sz),
            "noise_count": n_noise,
            "total_terms": n_total,
        })
pd.DataFrame(_sz_rows).to_csv(f'{RESULTS_DIR}clustering_cluster_sizes.csv', index=False)
print(f"Saved {len(_sz_rows)} cluster-size rows")


### 5.2 Pairwise (Cosine Similarity Thresholding)

Uses `model_utils_pairwise.evaluate_pairwise()` at the best threshold for each model.

In [8]:

# ── 5.2  PAIRWISE ────────────────────────────────────────────────────────────
pairwise_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    term_embeddings = embedding_cache[model_name]["norm_t"]
    threshold = BEST_PAIRWISE_THRESHOLDS[model_name]

    # Cosine similarity via mup.compute_similarities()
    sims = mup.compute_similarities(term_embeddings, term_to_idx, terms1, terms2)

    # Evaluate via mup.evaluate_pairwise() — returns sklearn metrics + MCC
    p, r, f1, pos_acc, neg_acc, mcc = mup.evaluate_pairwise(sims, labels, threshold)

    preds = (sims > threshold).astype(np.int8)

    pairwise_rows.append({
        "Model": config.MODELS[model_name]["display_name"],
        "Threshold": f"{threshold:.2f}",
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
        "MCC": f"{mcc:.4f}",
    })

    preds_store["pairwise"][model_name] = {"preds": preds, "similarities": sims}

display(Markdown("### Results — Pairwise"))
display(pd.DataFrame(pairwise_rows).set_index("Model"))


### Results — Pairwise

,Threshold,Precision,Recall,F1,Pos Acc,Neg Acc,MCC
Model,,,,,,,
All-MPNet-Base-v2,0.69,0.7365,0.4811,0.5820,0.4811,0.9995,0.5943
MPNet-Personality,0.66,0.7070,0.4464,0.5472,0.4464,0.9995,0.5608
SciBERT (SciVocab),0.89,0.4797,0.2575,0.3351,0.2575,0.9992,0.3501
BERT Base,0.90,0.3911,0.1522,0.2191,0.1522,0.9993,0.2427


### 5.3 Seeded Clustering

Uses `model_utils_seed.sample_unique_terms()` and `model_utils_seed.seed_clustering()` with the best number of seeds and threshold.

In [9]:
# ── 5.3  SEEDED CLUSTERING ────────────────────────────────────────────────────
seeded_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    norm_t = embedding_cache[model_name]["norm_t"].cpu().detach()
    term_to_embedding = {t: e for t, e in zip(unique_terms, norm_t)}
    params = BEST_SEEDED_PARAMS[model_name]

    # Sample initial seeds via mus.sample_unique_terms()
    initial_seeds, remaining_terms = mus.sample_unique_terms(
        params["n_initial_seeds"], full_df, np.array(unique_terms), random_state=config.SEED
    )

    # Run seeded clustering via mus.seed_clustering() (suppress verbose output)
    with redirect_stdout(StringIO()):
        seeds_cluster = mus.seed_clustering(
            initial_seeds, remaining_terms, term_to_embedding,
            threshold=params["threshold"],
        )

    # Evaluate via mus.evaluate_seed_clustering() — returns sklearn metrics
    p, r, f1, pos_acc, neg_acc = mus.evaluate_seed_clustering(
        seeds_cluster, full_df, np.array(unique_terms)
    )

    # Also extract raw predictions for audit reuse
    term_to_cluster = {}
    for cid, (seed, members) in enumerate(seeds_cluster.items()):
        for term in members:
            term_to_cluster[term] = cid

    c1 = np.array([term_to_cluster.get(t, -1) for t in terms1])
    c2 = np.array([term_to_cluster.get(t, -1) for t in terms2])
    preds = ((c1 == c2) & (c1 != -1)).astype(int)

    seeded_rows.append({
        "Model": config.MODELS[model_name]["display_name"],
        "Seeds": params["n_initial_seeds"], "Threshold": f"{params['threshold']:.2f}",
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "Pos Acc": f"{pos_acc:.4f}", "Neg Acc": f"{neg_acc:.4f}",
    })

    preds_store["seeded"][model_name] = {"preds": preds, "term_to_cluster": term_to_cluster}

display(Markdown("### Results — Seeded Clustering"))
display(pd.DataFrame(seeded_rows).set_index("Model"))


### Results — Seeded Clustering

,Seeds,Threshold,Precision,Recall,F1,Pos Acc,Neg Acc
Model,,,,,,,
All-MPNet-Base-v2,250,0.70,0.7587,0.4624,0.5746,0.4624,0.9996
MPNet-Personality,10,0.70,0.7341,0.3937,0.5125,0.3937,0.9996
SciBERT (SciVocab),250,0.85,0.1360,0.3648,0.1982,0.3648,0.9932
BERT Base,250,0.85,0.1558,0.3031,0.2058,0.3031,0.9952


In [ ]:
# ── Export A: primary performance ─────────────────────────────────────────────
_rows = []
for tech_name, rows_list in [("Clustering", clustering_rows),
                               ("Pairwise",  pairwise_rows),
                               ("Seeded",    seeded_rows)]:
    for row in rows_list:
        _rows.append({
            "technique": tech_name,
            "model":     row["Model"],
            "precision": float(row["Precision"]),
            "recall":    float(row["Recall"]),
            "f1":        float(row["F1"]),
        })
_perf_df = pd.DataFrame(_rows)
_perf_df.to_csv(f'{RESULTS_DIR}primary_performance.csv', index=False)
display(_perf_df)


In [ ]:
# ── Export B: confusion matrices ──────────────────────────────────────────────
_cm_rows = []
for tech_key, tech_name in [("clustering","Clustering"),("pairwise","Pairwise"),("seeded","Seeded")]:
    for model_name in MODELS_TO_EVAL:
        preds_info = preds_store[tech_key].get(model_name)
        if preds_info is None:
            continue
        p = preds_info["preds"]
        _cm_rows.append({
            "technique": tech_name,
            "model": config.MODELS[model_name]["display_name"],
            "tp": int(((p==1)&(labels==1)).sum()),
            "fp": int(((p==1)&(labels==0)).sum()),
            "fn": int(((p==0)&(labels==1)).sum()),
            "tn": int(((p==0)&(labels==0)).sum()),
        })
_cm_df = pd.DataFrame(_cm_rows)
_cm_df.to_csv(f'{RESULTS_DIR}confusion_matrices.csv', index=False)
display(_cm_df)


In [ ]:
# ── Export I: example pairs for supplementary figures ─────────────────────────
def _jaccard3(a, b):
    ga = set(a[i:i+3] for i in range(len(a)-2))
    gb = set(b[i:i+3] for i in range(len(b)-2))
    if not ga and not gb:
        return 0.0
    return len(ga & gb) / len(ga | gb)

_jaccard_scores = np.array([
    _jaccard3(str(t1), str(t2))
    for t1, t2 in zip(full_df['term1'], full_df['term2'])
])

_pos_mask = labels == 1
_neg_mask = labels == 0
_hard_neg = _neg_mask & (_jaccard_scores > 0.5)
_hard_pos = _pos_mask & (_jaccard_scores == 0.0)

# S11: Lexical trap fooled / resisted
for tech_key, tech_name in [("clustering","Clustering"),("pairwise","Pairwise"),("seeded","Seeded")]:
    for model_name in MODELS_TO_EVAL:
        info = preds_store[tech_key].get(model_name)
        if info is None:
            continue
        p = info["preds"]
        dname = config.MODELS[model_name]["display_name"].replace(" ","_").replace("(","").replace(")","").replace("/","_")

        fooled_idx = np.where(_hard_neg & (p == 1))[0]
        _df_fooled = full_df.iloc[fooled_idx][['term1','term2']].copy()
        _df_fooled['jaccard'] = _jaccard_scores[fooled_idx]
        _df_fooled['model'] = config.MODELS[model_name]["display_name"]
        _df_fooled['technique'] = tech_name
        if tech_key == "pairwise":
            _df_fooled['cosine_sim'] = info["similarities"][fooled_idx]
        _df_fooled.to_csv(f'{RESULTS_DIR}s11_lexical_trap_fooled_{dname}_{tech_name}.csv', index=False)

        resisted_idx = np.where(_hard_neg & (p == 0))[0]
        _df_resisted = full_df.iloc[resisted_idx][['term1','term2']].copy()
        _df_resisted['jaccard'] = _jaccard_scores[resisted_idx]
        _df_resisted['model'] = config.MODELS[model_name]["display_name"]
        _df_resisted['technique'] = tech_name
        if tech_key == "pairwise":
            _df_resisted['cosine_sim'] = info["similarities"][resisted_idx]
        _df_resisted.head(50).to_csv(f'{RESULTS_DIR}s11_lexical_trap_resisted_{dname}_{tech_name}.csv', index=False)

# S12: Semantic decay missed / caught
for tech_key, tech_name in [("clustering","Clustering"),("pairwise","Pairwise"),("seeded","Seeded")]:
    for model_name in MODELS_TO_EVAL:
        info = preds_store[tech_key].get(model_name)
        if info is None:
            continue
        p = info["preds"]
        dname = config.MODELS[model_name]["display_name"].replace(" ","_").replace("(","").replace(")","").replace("/","_")

        missed_idx = np.where(_hard_pos & (p == 0))[0]
        _df_missed = full_df.iloc[missed_idx][['term1','term2']].copy()
        if tech_key == "pairwise":
            _df_missed['cosine_sim'] = info["similarities"][missed_idx]
        _df_missed['model'] = config.MODELS[model_name]["display_name"]
        _df_missed['technique'] = tech_name
        _df_missed.to_csv(f'{RESULTS_DIR}s12_semantic_decay_missed_{dname}_{tech_name}.csv', index=False)

        caught_idx = np.where(_hard_pos & (p == 1))[0]
        _df_caught = full_df.iloc[caught_idx][['term1','term2']].copy()
        if tech_key == "pairwise":
            _df_caught['cosine_sim'] = info["similarities"][caught_idx]
        _df_caught['model'] = config.MODELS[model_name]["display_name"]
        _df_caught['technique'] = tech_name
        _df_caught.to_csv(f'{RESULTS_DIR}s12_semantic_decay_caught_{dname}_{tech_name}.csv', index=False)

# S14: Cross-model agreement (pairwise technique)
_all_pw_preds = np.column_stack([
    preds_store["pairwise"][m]["preds"] for m in MODELS_TO_EVAL
    if m in preds_store["pairwise"]
])
_pos_preds = _all_pw_preds[_pos_mask]
_pos_terms = full_df[_pos_mask][['term1','term2']].reset_index(drop=True)
_n_correct = _pos_preds.sum(axis=1)
_agreement_df = _pos_terms.copy()
_agreement_df['n_correct'] = _n_correct
_agreement_df['category'] = np.where(_n_correct == 4, 'unanimous_tp',
                             np.where(_n_correct == 0, 'unanimous_fn', 'disagreement'))
_agreement_df.to_csv(f'{RESULTS_DIR}s14_cross_model_agreement.csv', index=False)

# S16: Cosine distributions (All-MPNet-Base-v2, pairwise)
_best_key = "all-mpnet-base-v2"
if _best_key in preds_store["pairwise"]:
    _bp = preds_store["pairwise"][_best_key]["preds"]
    _bs = preds_store["pairwise"][_best_key]["similarities"]
    _outcome = np.where((labels==1)&(_bp==1),'TP',
               np.where((labels==0)&(_bp==1),'FP',
               np.where((labels==1)&(_bp==0),'FN','TN')))
    pd.DataFrame({'cosine_sim': _bs, 'outcome': _outcome}).to_csv(
        f'{RESULTS_DIR}s16_cosine_distributions.csv', index=False)

# S17: PR curves + calibrated thresholds (pairwise, all models)
from sklearn.metrics import precision_recall_curve
for model_name in MODELS_TO_EVAL:
    info = preds_store["pairwise"].get(model_name)
    if info is None:
        continue
    dname = config.MODELS[model_name]["display_name"].replace(" ","_").replace("(","").replace(")","").replace("/","_")
    prec_arr, rec_arr, thr_arr = precision_recall_curve(labels, info["similarities"])
    pd.DataFrame({'precision': prec_arr[:-1], 'recall': rec_arr[:-1], 'threshold': thr_arr}).to_csv(
        f'{RESULTS_DIR}s17_pr_curve_{dname}.csv', index=False)
    pd.DataFrame({'calibrated_threshold': [BEST_PAIRWISE_THRESHOLDS[model_name]]}).to_csv(
        f'{RESULTS_DIR}s17_calibrated_threshold_{dname}.csv', index=False)

# S18: Error overlap (all-mpnet-base-v2)
_fn_sets = {}
for tech_key in ["clustering", "pairwise", "seeded"]:
    info = preds_store[tech_key].get(_best_key)
    if info:
        _fn_sets[tech_key] = set(np.where(_pos_mask & (info["preds"] == 0))[0])

if len(_fn_sets) == 3:
    cl_s, pw_s, sd_s = _fn_sets["clustering"], _fn_sets["pairwise"], _fn_sets["seeded"]
    pd.DataFrame([{
        'all_three':       len(cl_s & pw_s & sd_s),
        'clust_pair_only': len((cl_s & pw_s) - sd_s),
        'clust_seed_only': len((cl_s & sd_s) - pw_s),
        'pair_seed_only':  len((pw_s & sd_s) - cl_s),
        'clust_only':      len(cl_s - pw_s - sd_s),
        'pair_only':       len(pw_s - cl_s - sd_s),
        'seed_only':       len(sd_s - cl_s - pw_s),
    }]).to_csv(f'{RESULTS_DIR}s18_error_overlap_counts.csv', index=False)
    _all3_idx = list(cl_s & pw_s & sd_s)
    full_df.iloc[_all3_idx][['term1','term2']].sample(
        min(20, len(_all3_idx)), random_state=42
    ).to_csv(f'{RESULTS_DIR}s18_error_overlap_irreducible.csv', index=False)

print("✓ All supplementary exports complete")


---

## 6. Audit 1 — Reliability (Stability Test)

**Objective:** Measure the Standard Error of Model (SEM) under stochastic embedding noise.

**Method:**
1. For each noise level $\sigma_\epsilon \in \{0, 0.05, 0.1, 0.2, 0.3\}$, inject Gaussian noise: $\tilde{e} = e + \sigma_\epsilon \cdot \hat\sigma \cdot z$, where $\hat\sigma$ is the per-dimension std and $z \sim \mathcal{N}(0, 1)$.
2. Re-run the technique $k = 5$ times per noise level.
3. Compute $\text{ICC}(1,1)$ across runs and $\text{SEM} = \text{SD}_{F_1} \cdot \sqrt{1 - \text{ICC}}$.

Lower SEM indicates a more stable model.

In [10]:
from sklearn.preprocessing import normalize

def _run_reliability_for_technique(technique, model_name, emb_cache,
                                   noise_levels, n_runs):
    """Run reliability audit for one model × one technique."""
    emb_raw = emb_cache["raw_np"]
    sigma   = emb_cache["sigma"]

    params_c = BEST_CLUSTERING_PARAMS.get(model_name, {})
    thr_pw   = BEST_PAIRWISE_THRESHOLDS.get(model_name, 0.5)
    params_s = BEST_SEEDED_PARAMS.get(model_name, {})

    results = {}
    for nl in noise_levels:
        f1_runs, all_preds = [], []
        for run in range(n_runs):
            rng = np.random.default_rng(config.SEED + run + int(nl * 10000))
            noisy = emb_raw + (rng.normal(0, 1, emb_raw.shape) * (nl * sigma)) if nl > 0 else emb_raw
            noisy_norm_np = normalize(noisy, norm="l2", axis=1)
            noisy_norm_np = np.ascontiguousarray(noisy_norm_np, dtype=np.float64)
            noisy_norm_t  = torch.nn.functional.normalize(
                torch.tensor(noisy, dtype=torch.float32), p=2, dim=1
            )

            if technique == "clustering":
                # ── Use muc.reduce_embeddings_with_umap + muc.run_hdbscan_clustering
                #    + muc.evaluate_clustering (same functions as Section 5.1)
                n_comp = params_c.get("n_components")
                if n_comp is None or n_comp >= noisy_norm_np.shape[1]:
                    red = noisy_norm_np
                else:
                    red, _ = muc.reduce_embeddings_with_umap(
                        noisy_norm_np, n_comp, random_state=config.SEED + run
                    )
                cl = muc.run_hdbscan_clustering(
                    red, params_c["min_cluster_size"], params_c["min_samples"]
                )
                _, _, f1_val, _, _, _, _ = muc.evaluate_clustering(
                    cl, full_df, np.array(unique_terms)
                )
                # Reconstruct preds for ICC computation
                p_ = ((cl[idx1] != -1) & (cl[idx1] == cl[idx2])).astype(int)

            elif technique == "pairwise":
                # ── Use mup.compute_similarities + mup.evaluate_pairwise
                #    (same functions as Section 5.2)
                sims = mup.compute_similarities(noisy_norm_t, term_to_idx, terms1, terms2)
                _, _, f1_val, _, _, _ = mup.evaluate_pairwise(sims, labels, thr_pw)
                p_ = (sims > thr_pw).astype(np.int8)

            else:  # seeded
                # ── Use mus.sample_unique_terms + mus.seed_clustering
                #    + mus.evaluate_seed_clustering (same functions as Section 5.3)
                nte = noisy_norm_t.cpu().detach()
                t2e = {t: e for t, e in zip(unique_terms, nte)}
                seeds, rem = mus.sample_unique_terms(
                    params_s.get("n_initial_seeds", 250), full_df,
                    np.array(unique_terms), random_state=config.SEED + run
                )
                with redirect_stdout(StringIO()):
                    sc = mus.seed_clustering(
                        seeds, rem, t2e, threshold=params_s.get("threshold", 0.7)
                    )
                _, _, f1_val, _, _ = mus.evaluate_seed_clustering(
                    sc, full_df, np.array(unique_terms)
                )
                # Reconstruct preds for ICC computation
                t2c = {}
                for cid, (sd, ms) in enumerate(sc.items()):
                    for m in ms:
                        t2c[m] = cid
                c1 = np.array([t2c.get(t, -1) for t in terms1])
                c2 = np.array([t2c.get(t, -1) for t in terms2])
                p_ = ((c1 == c2) & (c1 != -1)).astype(int)

            f1_runs.append(f1_val)
            all_preds.append(p_)

        f1_arr = np.array(f1_runs)
        preds_mat = np.array(all_preds)
        icc = shared.compute_icc_oneway(
            preds_mat.sum(axis=0), (preds_mat ** 2).sum(axis=0),
            preds_mat.shape[1], n_runs
        )
        sd = np.std(f1_arr, ddof=1)
        sem = sd * np.sqrt(max(0, 1 - icc))
        results[nl] = {"mean_f1": f1_arr.mean(), "sd_f1": sd, "icc": icc, "sem": sem}
    return results


# ── Run for all techniques ───────────────────────────────────────────────────
audit1_records = []
for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        if model_name not in embedding_cache:
            continue
        print(f"  Reliability | {technique:10s} | {model_name} ...", end="", flush=True)
        res = _run_reliability_for_technique(
            technique, model_name, embedding_cache[model_name],
            NOISE_LEVELS, N_RELIABILITY_RUNS
        )
        print(" ✓")
        for nl, vals in res.items():
            audit1_records.append({
                "Technique": technique,
                "Model": config.MODELS[model_name]["display_name"],
                "Noise σ": nl,
                "Mean F1": f"{vals['mean_f1']:.4f}",
                "SD(F1)": f"{vals['sd_f1']:.4f}",
                "ICC": f"{vals['icc']:.4f}",
                "SEM": f"{vals['sem']:.4f}",
            })

df_audit1 = pd.DataFrame(audit1_records)
display(Markdown("### Audit 1 — Reliability Results"))

for tech in ["clustering", "pairwise", "seeded"]:
    display(Markdown(f"#### {tech.upper()}"))
    sub = df_audit1[df_audit1["Technique"] == tech].drop(columns="Technique")

    # Full table: all metrics per model × noise level
    display(sub.set_index(["Model", "Noise σ"]))

    # Compact pivot: Mean F1 per model × noise level
    display(Markdown("*Mean F1 pivot (rows = Model, cols = Noise σ):*"))
    pivot = sub.pivot(index="Model", columns="Noise σ", values="Mean F1")
    display(pivot)


  Reliability | clustering | all-mpnet-base-v2 ... ✓
  Reliability | clustering | dwulff/mpnet-personality ... ✓
  Reliability | clustering | allenai/scibert_scivocab_uncased ... ✓
  Reliability | clustering | bert-base-uncased ... ✓
  Reliability | pairwise   | all-mpnet-base-v2 ... ✓
  Reliability | pairwise   | dwulff/mpnet-personality ... ✓
  Reliability | pairwise   | allenai/scibert_scivocab_uncased ... ✓
  Reliability | pairwise   | bert-base-uncased ... ✓
  Reliability | seeded     | all-mpnet-base-v2 ... ✓
  Reliability | seeded     | dwulff/mpnet-personality ... ✓
  Reliability | seeded     | allenai/scibert_scivocab_uncased ... ✓
  Reliability | seeded     | bert-base-uncased ... ✓


### Audit 1 — Reliability Results

#### CLUSTERING

Mean F1  SD(F1)     ICC     SEM
Model              Noise σ                                
All-MPNet-Base-v2  0.00     0.5859  0.0000  1.0000  0.0000
                   0.05     0.5840  0.0027  0.9776  0.0004
                   0.10     0.5838  0.0057  0.9655  0.0011
                   0.20     0.5851  0.0050  0.9382  0.0013
                   0.30     0.5923  0.0082  0.9076  0.0025
MPNet-Personality  0.00     0.5725  0.0000  1.0000  0.0000
                   0.05     0.5720  0.0025  0.9825  0.0003
                   0.10     0.5712  0.0025  0.9680  0.0005
                   0.20     0.5704  0.0042  0.9442  0.0010
                   0.30     0.5809  0.0053  0.9171  0.0015
SciBERT (SciVocab) 0.00     0.3951  0.0000  1.0000  0.0000
                   0.05     0.3936  0.0033  0.9689  0.0006
                   0.10     0.3898  0.0019  0.9479  0.0004
                   0.20     0.3891  0.0040  0.9013  0.0012
                   0.30     0.3846  0.0069  0.8620  0.0026
BERT Base          0.00     0.3568  0.0000  1.0000  0.0000
                   0.05     0.3605  0.0022  0.9652  0.0004
                   0.10     0.3643  0.0077  0.9197  0.0022
                   0.20     0.3493  0.0550  0.6955  0.0304
                   0.30     0.3621  0.0446  0.7135  0.0239

*Mean F1 pivot (rows = Model, cols = Noise σ):*

...

Noise σ,0.00,0.05,0.10,0.20,0.30
Model,,,,,
All-MPNet-Base-v2,0.5820,0.5833,0.5780,0.5540,0.5077
BERT Base,0.2191,0.2163,0.2086,0.1597,0.0930
MPNet-Personality,0.5472,0.5456,0.5428,0.5272,0.4835
SciBERT (SciVocab),0.3351,0.3362,0.3330,0.3164,0.2671


#### SEEDED

Mean F1  SD(F1)     ICC     SEM
Model              Noise σ                                
All-MPNet-Base-v2  0.00     0.5728  0.0050  0.9318  0.0013
                   0.05     0.5721  0.0053  0.9277  0.0014
                   0.10     0.5707  0.0062  0.9192  0.0018
                   0.20     0.5459  0.0045  0.9170  0.0013
                   0.30     0.5151  0.0068  0.9142  0.0020
MPNet-Personality  0.00     0.5125  0.0007  0.9965  0.0000
                   0.05     0.5104  0.0028  0.9879  0.0003
                   0.10     0.5034  0.0027  0.9753  0.0004
                   0.20     0.4766  0.0071  0.9435  0.0017
                   0.30     0.4220  0.0028  0.9464  0.0007
SciBERT (SciVocab) 0.00     0.1851  0.0213  0.4234  0.0162
                   0.05     0.1822  0.0207  0.4177  0.0158
                   0.10     0.1844  0.0232  0.4151  0.0177
                   0.20     0.1710  0.0242  0.4255  0.0183
                   0.30     0.1635  0.0221  0.5632  0.0146
BERT Base          0.00     0.2034  0.0318  0.4916  0.0227
                   0.05     0.2023  0.0264  0.4863  0.0189
                   0.10     0.2017  0.0276  0.4825  0.0198
                   0.20     0.1842  0.0319  0.5172  0.0221
                   0.30     0.1728  0.0262  0.5416  0.0178

*Mean F1 pivot (rows = Model, cols = Noise σ):*

Noise σ,0.00,0.05,0.10,0.20,0.30
Model,,,,,
All-MPNet-Base-v2,0.5728,0.5721,0.5707,0.5459,0.5151
BERT Base,0.2034,0.2023,0.2017,0.1842,0.1728
MPNet-Personality,0.5125,0.5104,0.5034,0.4766,0.4220
SciBERT (SciVocab),0.1851,0.1822,0.1844,0.1710,0.1635


In [ ]:
# ── Export D: audit 1 stability ───────────────────────────────────────────────
_a1 = pd.DataFrame(audit1_records).rename(columns={
    "Technique": "technique", "Model": "model",
    "Noise σ": "noise_level", "Mean F1": "mean_f1",
    "SD(F1)": "sd_f1", "ICC": "icc", "SEM": "sem",
})
for col in ["mean_f1", "sd_f1", "icc", "sem"]:
    _a1[col] = _a1[col].astype(float)
_a1.to_csv(f'{RESULTS_DIR}audit1_stability.csv', index=False)
display(_a1.head(10))


---

## 7. Audit 2 — Discriminant Validity (Lexical Trap)

**Objective:** Detect whether models are fooled by surface-level orthographic similarity.

We evaluate models on two types of negative pairs (label = 0 only):

1. **Hard Negatives (Lexical Traps)**: Negative pairs whose character 3-gram Jaccard similarity exceeds 0.5 (high surface similarity but semantically unrelated).
2. **Easy Negatives**: Negative pairs with zero lexical overlap (Jaccard = 0.0).

Because both subsets contain **only label=0 pairs**, Precision / Recall / F1 (which treat label=1 as positive) are undefined and always 0. The sole meaningful metric is:

- **FPR** (False Positive Rate): $\text{FPR} = \frac{FP}{FP + TN}$

Lower FPR on hard negatives means the model correctly rejects unrelated terms despite high letter overlap (good discriminant validity).


In [ ]:
audit2_records = []

for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        info = preds_store[technique].get(model_name)
        if info is None:
            continue
        preds = info["preds"]

        hard_metrics = shared.compute_subset_metrics(labels, preds, hard_neg_mask)
        easy_metrics = shared.compute_subset_metrics(labels, preds, easy_neg_mask)

        fpr_hard = hard_metrics['fpr']
        fpr_easy = easy_metrics['fpr']
        delta = fpr_hard - fpr_easy
        retention = fpr_easy / fpr_hard if fpr_hard > 0 else float("nan")

        audit2_records.append({
            "Technique": technique,
            "Model": config.MODELS[model_name]["display_name"],
            "FPR (Hard Neg)": f"{fpr_hard:.4f}",
            "FPR (Easy Neg)": f"{fpr_easy:.4f}",
            "ΔFPR": f"{delta:+.4f}",
            "Retention Rate": f"{retention:.4f}" if not np.isnan(retention) else "NaN",
            "N Hard": int(hard_metrics['n_subset']),
            "N Easy": int(easy_metrics['n_subset']),
        })

display(Markdown("### Audit 2 — Discriminant Validity Results"))
display(Markdown("""
**Hard Negatives (Lexical Traps)** are negative pairs with high surface similarity (Jaccard > 0.5).  
**Easy Negatives** are negative pairs with zero lexical overlap (Jaccard = 0.0).  

> **Note:** These subsets contain only label=0 pairs (true negatives), so Precision / Recall / F1  
> (which treat label=1 as positive) are undefined and always 0. Only **FPR** is the meaningful metric here:  
> lower FPR means the model correctly rejects more lexically-similar but unrelated pairs.  
> **ΔFPR** = FPR(Hard) − FPR(Easy); positive values indicate greater confusion on lexical traps.  
> **Retention Rate** = FPR(Easy) / FPR(Hard); closer to 1.0 means similar error rates on both subsets
> (model fails equally on hard and easy); closer to 0 means the model is especially confused by lexical traps.
"""))
display(pd.DataFrame(audit2_records).set_index(["Technique", "Model"]))

In [ ]:
# ── Export E: audit 2 lexical trap ────────────────────────────────────────────
_a2 = pd.DataFrame(audit2_records).rename(columns={
    "Technique": "technique", "Model": "model",
    "FPR (Hard Neg)": "hard_neg_fpr", "FPR (Easy Neg)": "easy_neg_fpr",
    "ΔFPR": "delta_fpr", "Retention Rate": "retention_rate",
    "N Hard": "n_hard", "N Easy": "n_easy",
})
for col in ["hard_neg_fpr", "easy_neg_fpr", "delta_fpr"]:
    _a2[col] = pd.to_numeric(_a2[col], errors='coerce')
_a2.to_csv(f'{RESULTS_DIR}audit2_lexical_trap.csv', index=False)
display(_a2)


---

## 8. Audit 3 — Differential Item Functioning (Rare-Word Bias)

**Objective:** Detect systematic bias against rare / technical terminology.

$$\Delta\text{Recall} = \text{Recall}_{\text{Common}} - \text{Recall}_{\text{Rare}}$$

- **Common pairs**: top 10 % of Zipf frequency.  
- **Rare pairs**: bottom 10 % of Zipf frequency.  
- A large positive $\Delta$Recall indicates the model underperforms on rare terms.

In [11]:
if not has_wordfreq:
    display(Markdown("**⚠ Skipped** — `wordfreq` library is not installed."))
else:
    audit3_records = []
    for technique in ["clustering", "pairwise", "seeded"]:
        for model_name in MODELS_TO_EVAL:
            preds = preds_store[technique].get(model_name, {}).get("preds")
            if preds is None:
                continue
            rec_common = shared.recall_on_subset(labels, preds, common_mask)
            rec_rare   = shared.recall_on_subset(labels, preds, rare_mask)
            gap = rec_common - rec_rare
            retention = rec_rare / rec_common if rec_common > 0 else float("nan")
            audit3_records.append({
                "Technique": technique,
                "Model": config.MODELS[model_name]["display_name"],
                "Recall (Common)": f"{rec_common:.4f}",
                "Recall (Rare)": f"{rec_rare:.4f}",
                "ΔRecall": f"{gap:+.4f}",
                "Retention Rate": f"{retention:.4f}" if not np.isnan(retention) else "NaN",
                "N Common": int(common_mask.sum()),
                "N Rare": int(rare_mask.sum()),
            })

    display(Markdown("### Audit 3 — DIF Results"))
    display(pd.DataFrame(audit3_records).set_index(["Technique", "Model"]))

### Audit 3 — DIF Results

Recall (Common) Recall (Rare)  ΔRecall  \
Technique  Model                                                       
clustering All-MPNet-Base-v2           0.4615        0.4103  +0.0513   
           MPNet-Personality           0.4744        0.3910  +0.0833   
           SciBERT (SciVocab)          0.2628        0.2051  +0.0577   
           BERT Base                   0.3333        0.1731  +0.1603   
pairwise   All-MPNet-Base-v2           0.5064        0.3141  +0.1923   
           MPNet-Personality           0.4487        0.2692  +0.1795   
           SciBERT (SciVocab)          0.2372        0.1410  +0.0962   
           BERT Base                   0.1410        0.0833  +0.0577   
seeded     All-MPNet-Base-v2           0.4359        0.3205  +0.1154   
           MPNet-Personality           0.3718        0.2372  +0.1346   
           SciBERT (SciVocab)          0.3526        0.2308  +0.1218   
           BERT Base                   0.2821        0.1346  +0.1474   

                              Retention Rate  N Common  N Rare  
Technique  Model                                                
clustering All-MPNet-Base-v2          0.8889       156     156  
           MPNet-Personality          0.8243       156     156  
           SciBERT (SciVocab)         0.7805       156     156  
           BERT Base                  0.5192       156     156  
pairwise   All-MPNet-Base-v2          0.6203       156     156  
           MPNet-Personality          0.6000       156     156  
           SciBERT (SciVocab)         0.5946       156     156  
           BERT Base                  0.5909       156     156  
seeded     All-MPNet-Base-v2          0.7353       156     156  
           MPNet-Personality          0.6379       156     156  
           SciBERT (SciVocab)         0.6545       156     156  
           BERT Base                  0.4773       156     156

In [ ]:
# ── Export F: audit 3 rare-word bias ──────────────────────────────────────────
if 'audit3_records' in dir() and audit3_records:
    _a3 = pd.DataFrame(audit3_records).rename(columns={
        "Technique": "technique", "Model": "model",
        "Recall (Common)": "recall_common", "Recall (Rare)": "recall_rare",
        "ΔRecall": "delta_recall", "Retention Rate": "retention_rate",
        "N Common": "n_common", "N Rare": "n_rare",
    })
    for col in ["recall_common", "recall_rare", "delta_recall"]:
        _a3[col] = pd.to_numeric(_a3[col], errors='coerce')
    _a3.to_csv(f'{RESULTS_DIR}audit3_rare_word.csv', index=False)
    display(_a3)
else:
    print("⚠ audit3 skipped — wordfreq not installed")


---

## 9. Audit 4 — Semantic Decay (Semantic Gap Test)

**Objective:** Test robustness as keyword overlap vanishes.

$$\Delta\text{Recall} = \text{Recall}_{\text{Hard}} - \text{Recall}_{\text{Easy}}$$

- **Easy positives**: Jaccard > 0.5 (lexical anchors — the model can "cheat" with surface cues).  
- **Hard positives**: Jaccard = 0.0 (semantic gap — requires genuine semantic understanding).  
- A negative ΔRecall indicates performance degrades on purely-semantic pairs.

**Retention Rate** = Recall(Hard) / Recall(Easy): scale-free complement to ΔRecall — answers "what fraction of easy-pair recall survives on hard pairs?" independently of the model's absolute performance level. Closer to 1.0 = less semantic decay.

In [12]:
audit4_records = []

for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        preds = preds_store[technique].get(model_name, {}).get("preds")
        if preds is None:
            continue
        rec_easy = shared.recall_on_subset(labels, preds, easy_mask)
        rec_hard = shared.recall_on_subset(labels, preds, hard_mask)
        delta = rec_hard - rec_easy
        retention = rec_hard / rec_easy if rec_easy > 0 else float("nan")
        audit4_records.append({
            "Technique": technique,
            "Model": config.MODELS[model_name]["display_name"],
            "Recall (Easy)": f"{rec_easy:.4f}",
            "Recall (Hard)": f"{rec_hard:.4f}",
            "ΔRecall": f"{delta:+.4f}",
            "Retention Rate": f"{retention:.4f}" if not np.isnan(retention) else "NaN",
            "N Easy": int(easy_mask.sum()),
            "N Hard": int(hard_mask.sum()),
        })

display(Markdown("### Audit 4 — Semantic Decay Results"))
display(pd.DataFrame(audit4_records).set_index(["Technique", "Model"]))

### Audit 4 — Semantic Decay Results

Recall (Easy) Recall (Hard)  ΔRecall  \
Technique  Model                                                     
clustering All-MPNet-Base-v2         0.8684        0.2708  -0.5976   
           MPNet-Personality         0.8816        0.2495  -0.6321   
           SciBERT (SciVocab)        0.6447        0.0600  -0.5848   
           BERT Base                 0.6053        0.0812  -0.5240   
pairwise   All-MPNet-Base-v2         0.8816        0.1683  -0.7133   
           MPNet-Personality         0.8816        0.1219  -0.7597   
           SciBERT (SciVocab)        0.5724        0.0329  -0.5395   
           BERT Base                 0.4079        0.0426  -0.3653   
seeded     All-MPNet-Base-v2         0.8289        0.1934  -0.6355   
           MPNet-Personality         0.8355        0.1199  -0.7156   
           SciBERT (SciVocab)        0.7303        0.1044  -0.6258   
           BERT Base                 0.6118        0.0986  -0.5132   

                              Retention Rate  N Easy  N Hard  
Technique  Model                                              
clustering All-MPNet-Base-v2          0.3118     152     517  
           MPNet-Personality          0.2830     152     517  
           SciBERT (SciVocab)         0.0930     152     517  
           BERT Base                  0.1342     152     517  
pairwise   All-MPNet-Base-v2          0.1909     152     517  
           MPNet-Personality          0.1382     152     517  
           SciBERT (SciVocab)         0.0574     152     517  
           BERT Base                  0.1043     152     517  
seeded     All-MPNet-Base-v2          0.2333     152     517  
           MPNet-Personality          0.1435     152     517  
           SciBERT (SciVocab)         0.1430     152     517  
           BERT Base                  0.1612     152     517

In [ ]:
# ── Export G: audit 4 semantic decay ──────────────────────────────────────────
_a4 = pd.DataFrame(audit4_records).rename(columns={
    "Technique": "technique", "Model": "model",
    "Recall (Easy)": "recall_easy", "Recall (Hard)": "recall_hard",
    "ΔRecall": "delta_recall", "Retention Rate": "retention_rate",
    "N Easy": "n_easy", "N Hard": "n_hard",
})
for col in ["recall_easy", "recall_hard", "delta_recall"]:
    _a4[col] = pd.to_numeric(_a4[col], errors='coerce')
_a4.to_csv(f'{RESULTS_DIR}audit4_semantic_decay.csv', index=False)
display(_a4)


---

## 10. Audit 5 — Structural Validity (Map Match)

**Objective:** Assess whether model-derived distances correlate with expert-curated ELSST graph distances.

We compute three correlation coefficients:

| Correlation | Description |
|------------|-------------|
| **Spearman ρ** | Rank-order correlation (robust to non-linear monotonic relationships) |
| **Pearson r** | Linear correlation between distances |
| **Biserial r** | Correlation between a binary model variable (same cluster = 0, diff = 1) and continuous expert distance (handles imbalanced data) |

For **clustering / seeded**: binary distance (0 = same cluster, 1 = different).  
For **pairwise**: binary distance (0 = paired, 1 = not paired) and cosine distance.

In [13]:
expert_mask = shortest_path >= 0
d_expert = shortest_path[expert_mask].astype(float)
print(f"Pairs with valid expert distances: {expert_mask.sum():,}\n")

audit5_records = []

# ── Clustering & Seeded: binary model distance ──────────────────────────────
for technique in ["clustering", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        info = preds_store[technique].get(model_name)
        if info is None:
            continue

        if technique == "clustering":
            cl = info["cluster_labels"]
            l1, l2 = cl[info["idx1"]], cl[info["idx2"]]
            # 1 if same cluster (synonym), 0 if different
            d_model = np.where((l1 != -1) & (l1 == l2), 1, 0).astype(float)
        else:
            t2c = info["term_to_cluster"]
            c1 = np.array([t2c.get(t, -1) for t in terms1])
            c2 = np.array([t2c.get(t, -1) for t in terms2])
            # 1 if same cluster (synonym), 0 if different
            d_model = np.where((c1 != -1) & (c1 == c2), 1, 0).astype(float)

        d_m = d_model[expert_mask]
        rho_s = shared.spearman_corr(d_expert, d_m)
        rho_p, _ = shared.pearson_corr(d_expert, d_m)
        rb = shared.biserial_corr(d_m, d_expert)

        audit5_records.append({
            "Technique": technique,
            "Model": config.MODELS[model_name]["display_name"],
            "Metric": "binary_dist",
            "Spearman ρ": f"{rho_s:.4f}",
            "Pearson r": f"{rho_p:.4f}",
            "Biserial r": f"{rb:.4f}" if not np.isnan(rb) else "N/A",
        })

# ── Pairwise: multiple distance metrics ─────────────────────────────────────
for model_name in MODELS_TO_EVAL:
    info = preds_store["pairwise"].get(model_name)
    if info is None:
        continue
    preds = info["preds"]
    sims  = info["similarities"]

    # (a) Binary distance (1 if paired/synonym, 0 if not paired)
    d_model = preds.astype(float)
    d_m = d_model[expert_mask]

    rho_s  = shared.spearman_corr(d_expert, d_m)
    rho_p, _ = shared.pearson_corr(d_expert, d_m)
    rb = shared.biserial_corr(d_m, d_expert)

    audit5_records.append({
        "Technique": "pairwise", "Model": config.MODELS[model_name]["display_name"],
        "Metric": "binary_dist",
        "Spearman ρ": f"{rho_s:.4f}", "Pearson r": f"{rho_p:.4f}",
        "Biserial r": f"{rb:.4f}" if not np.isnan(rb) else "N/A",
    })

    # (b) Cosine distance
    cos_dist = 1.0 - sims
    rho_cd = shared.spearman_corr(d_expert, cos_dist[expert_mask])
    rp_cd, _ = shared.pearson_corr(d_expert, cos_dist[expert_mask])

    audit5_records.append({
        "Technique": "pairwise", "Model": config.MODELS[model_name]["display_name"],
        "Metric": "cosine_distance",
        "Spearman ρ": f"{rho_cd:.4f}", "Pearson r": f"{rp_cd:.4f}",
        "Biserial r": "N/A (continuous)",
    })

display(Markdown("### Audit 5 — Structural Validity Results"))
display(pd.DataFrame(audit5_records).set_index(["Technique", "Model", "Metric"]))

Pairs with valid expert distances: 241,280



### Audit 5 — Structural Validity Results

Spearman ρ Pearson r  \
Technique  Model              Metric                                 
clustering All-MPNet-Base-v2  binary_dist        -0.1042   -0.1361   
           MPNet-Personality  binary_dist        -0.1009   -0.1318   
           SciBERT (SciVocab) binary_dist        -0.0828   -0.1064   
           BERT Base          binary_dist        -0.0778   -0.0997   
seeded     All-MPNet-Base-v2  binary_dist        -0.0998   -0.1308   
           MPNet-Personality  binary_dist        -0.0923   -0.1208   
           SciBERT (SciVocab) binary_dist        -0.0894   -0.1060   
           BERT Base          binary_dist        -0.0678   -0.0846   
pairwise   All-MPNet-Base-v2  binary_dist        -0.1027   -0.1341   
                              cosine_distance     0.2388    0.2991   
           MPNet-Personality  binary_dist        -0.0995   -0.1294   
                              cosine_distance     0.1504    0.2463   
           SciBERT (SciVocab) binary_dist        -0.0800   -0.1013   
                              cosine_distance     0.1653    0.1867   
           BERT Base          binary_dist        -0.0534   -0.0680   
                              cosine_distance     0.1688    0.1905   

                                                     Biserial r  
Technique  Model              Metric                             
clustering All-MPNet-Base-v2  binary_dist               -0.7115  
           MPNet-Personality  binary_dist               -0.6892  
           SciBERT (SciVocab) binary_dist               -0.6654  
           BERT Base          binary_dist               -0.5915  
seeded     All-MPNet-Base-v2  binary_dist               -0.7212  
           MPNet-Personality  binary_dist               -0.7087  
           SciBERT (SciVocab) binary_dist               -0.4082  
           BERT Base          binary_dist               -0.3663  
pairwise   All-MPNet-Base-v2  binary_dist               -0.7203  
                              cosine_distance  N/A (continuous)  
           MPNet-Personality  binary_dist               -0.7108  
                              cosine_distance  N/A (continuous)  
           SciBERT (SciVocab) binary_dist               -0.6262  
                              cosine_distance  N/A (continuous)  
           BERT Base          binary_dist               -0.4958  
                              cosine_distance  N/A (continuous)

In [ ]:
# ── Export H: audit 5 structural validity ─────────────────────────────────────
_a5_all = pd.DataFrame(audit5_records).rename(columns={
    "Technique": "technique", "Model": "model", "Metric": "metric",
    "Spearman ρ": "spearman_r", "Pearson r": "pearson_r", "Biserial r": "biserial_r",
})
for col in ["spearman_r", "pearson_r"]:
    _a5_all[col] = pd.to_numeric(_a5_all[col], errors='coerce')

_a5_bin = _a5_all[_a5_all["metric"] == "binary_dist"].copy()
_a5_bin["biserial_r"] = pd.to_numeric(_a5_bin["biserial_r"], errors='coerce')
_a5_bin.to_csv(f'{RESULTS_DIR}audit5_structural.csv', index=False)

_a5_cos = _a5_all[
    (_a5_all["metric"] == "cosine_distance") & (_a5_all["technique"] == "pairwise")
][["model", "spearman_r", "pearson_r"]].copy()
_a5_cos.to_csv(f'{RESULTS_DIR}audit5_cosine_structural.csv', index=False)

display(_a5_bin)
display(_a5_cos)


In [ ]:
# ── Export J: structural alignment examples ────────────────────────────────────
_struct_rows = []
for dist in [1, 2, 3, 5]:
    _sub = full_df[full_df['shortest_path'] == dist].copy()
    if _best_key in preds_store['pairwise']:
        _sub = _sub.copy()
        _sub['cosine_sim'] = preds_store['pairwise'][_best_key]['similarities'][_sub.index]
    _struct_rows.append(_sub.sample(min(5, len(_sub)), random_state=42))
pd.concat(_struct_rows).to_csv(f'{RESULTS_DIR}s15_structural_examples.csv', index=False)
print("✓ Structural examples saved")


---

## 11. Consolidated Summary

A single overview table combining the core F1 metric across all models and techniques, plus a per-technique best-model highlight.

In [15]:
# ── Build consolidated F1 summary ─────────────────────────────────────────────
summary_rows = []

for tech_name, rows_list in [("Clustering", clustering_rows),
                              ("Pairwise", pairwise_rows),
                              ("Seeded", seeded_rows)]:
    for row in rows_list:
        summary_rows.append({
            "Technique": tech_name,
            "Model": row["Model"],
            "Precision": row["Precision"],
            "Recall": row["Recall"],
            "F1": row["F1"],
        })

df_summary = pd.DataFrame(summary_rows)

display(Markdown("### F1 Summary — All Models × All Techniques"))
pivot_f1 = df_summary.pivot(index="Model", columns="Technique", values="F1")
display(pivot_f1)

# ── Best model per technique ─────────────────────────────────────────────────
display(Markdown("### Best Model per Technique"))
for tech in ["Clustering", "Pairwise", "Seeded"]:
    sub = df_summary[df_summary["Technique"] == tech]
    best = sub.loc[sub["F1"].astype(float).idxmax()]
    print(f"  {tech:12s} → {best['Model']}  (F1 = {best['F1']})")

# ── Save summary to CSV ─────────────────────────────────────────────────────
import os
os.makedirs("results", exist_ok=True)
summary_path = "results/elsst/final_test_summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"\n✓ Summary saved to {summary_path}")

### F1 Summary — All Models × All Techniques

Technique,Clustering,Pairwise,Seeded
Model,,,
All-MPNet-Base-v2,0.5859,0.5820,0.5746
BERT Base,0.3568,0.2191,0.2058
MPNet-Personality,0.5725,0.5472,0.5125
SciBERT (SciVocab),0.3951,0.3351,0.1982


### Best Model per Technique

  Clustering   → All-MPNet-Base-v2  (F1 = 0.5859)
  Pairwise     → All-MPNet-Base-v2  (F1 = 0.5820)
  Seeded       → All-MPNet-Base-v2  (F1 = 0.5746)

✓ Summary saved to results/elsst/final_test_summary.csv


---

## Reproducibility Notes

| Item | Value |
|------|-------|
| Random seed | `SEED = 42` across all operations |
| Test data | `datasets/processed_datasets/test_{positive,negative}_pairs.csv` |
| Models | Loaded from local HuggingFace cache (`local_files_only=True`) |
| Hyperparameters | Fixed from prior cross-validation (Section 1) |
| Correlations | Spearman ρ, Pearson r, and Biserial r (Audit 5) |

**To reproduce:** Run all cells top-to-bottom in a fresh kernel.  
All utility functions live in `model_utils_shared.py`, `model_utils_clustering.py`, `model_utils_pairwise.py`, and `model_utils_seed.py`.